In [ ]:
import torch
import torch.nn as nn

# A single convolutional layer:
#   - 3 input channels (RGB)
#   - 16 output channels (= 16 learned filters)
#   - 5x5 spatial kernel
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5)

# Weight shape: (out_channels, in_channels, kH, kW)
print(conv.weight.shape)   # → torch.Size([16, 3, 5, 5])
print(conv.bias.shape)     # → torch.Size([16])  (one bias per filter)

# Forward pass on a batch of 8 images, each 3×64×64
x = torch.randn(8, 3, 64, 64)   # (batch, channels, H, W)
out = conv(x)
print(out.shape)   # → torch.Size([8, 16, 60, 60])  (no padding, stride=1)


In [ ]:
# Visualising the shapes through a small CNN
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1),   # 32 filters, 3x3
    nn.ReLU(),
    nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 64 filters, 3x3
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),        # halves H and W
)

x = torch.randn(1, 3, 64, 64)   # 1 image, RGB, 64x64
for layer in model:
    x = layer(x)
    print(f'{layer.__class__.__name__:20s}  →  {tuple(x.shape)}')

# Output:
# Conv2d                →  (1, 32, 64, 64)
# ReLU                  →  (1, 32, 64, 64)
# Conv2d                →  (1, 64, 64, 64)
# ReLU                  →  (1, 64, 64, 64)
# MaxPool2d             →  (1, 64, 32, 32)


In [ ]:
import torch
import torch.nn as nn

# Manually verifying the output size formula
# Input: N=32, kernel W=5, stride S=1, padding P=0
# Expected output: (32 - 5) / 1 + 1 = 28

conv_valid = nn.Conv2d(1, 1, kernel_size=5, stride=1, padding=0)  # 'valid'
conv_same  = nn.Conv2d(1, 1, kernel_size=5, stride=1, padding=2)  # 'same'

x = torch.randn(1, 1, 32, 32)  # Input: 1 image, 1 channel, 32x32

print(conv_valid(x).shape)   # → torch.Size([1, 1, 28, 28])
print(conv_same(x).shape)    # → torch.Size([1, 1, 32, 32])

# With stride=2, 'valid' convolution halves spatial size
conv_stride2 = nn.Conv2d(1, 1, kernel_size=3, stride=2, padding=0)
print(conv_stride2(x).shape) # → torch.Size([1, 1, 15, 15])
# formula: (32 - 3) / 2 + 1 = 15.5 → floor = 15


In [ ]:
import torch
import torch.nn as nn

# ReLU applied element-wise after convolution
conv = nn.Conv2d(3, 16, kernel_size=3, padding=1)
relu = nn.ReLU()

x = torch.randn(1, 3, 64, 64)
z = conv(x)      # pre-activation: can be any real value
a = relu(z)      # post-activation: non-negative

print(f'Pre-ReLU  min: {z.min():.2f}, max: {z.max():.2f}')
print(f'Post-ReLU min: {a.min():.2f}, max: {a.max():.2f}')  # min is 0

# Alternatively, use inplace=True to save memory
relu_inplace = nn.ReLU(inplace=True)

In [ ]:
import torch
import torch.nn as nn

pool = nn.MaxPool2d(kernel_size=2, stride=2)

# Demonstrate translation invariance within the pooling window
x1 = torch.tensor([[[[3., 0.],
                      [0., 0.]]]])   # strong activation top-left
x2 = torch.tensor([[[[0., 0.],
                      [0., 3.]]]])   # strong activation bottom-right

print(pool(x1))  # → tensor([[[[3.]]]])
print(pool(x2))  # → tensor([[[[3.]]]])  same output!

# In practice, pooling is applied across all channels independently
x = torch.randn(1, 64, 32, 32)
out = pool(x)
print(out.shape)   # → torch.Size([1, 64, 16, 16])  — halved spatially

In [ ]:
import torch
import torch.nn as nn

# Max pooling vs. strided convolution for downsampling

# Option 1: Max pooling (no learnable parameters)
downsample_pool = nn.Sequential(
    nn.Conv2d(64, 64, kernel_size=3, padding=1),
    nn.MaxPool2d(kernel_size=2, stride=2),
)

# Option 2: Strided convolution (learns how to downsample)
downsample_conv = nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1)

x = torch.randn(1, 64, 32, 32)
print(downsample_pool(x).shape)   # → (1, 64, 16, 16)
print(downsample_conv(x).shape)   # → (1, 64, 16, 16)
# Same output shape, but strided conv has learnable parameters

In [ ]:
import torch
import torch.nn as nn

gap = nn.AdaptiveAvgPool2d(output_size=(1, 1))   # Global Average Pooling

# Works for any spatial size
for h, w in [(7, 7), (3, 3), (14, 14)]:
    x = torch.randn(1, 1024, h, w)
    out = gap(x)
    print(f'Input: (1, 1024, {h}, {w})  →  GAP output: {tuple(out.shape)}')

# All produce: (1, 1024, 1, 1) — then flatten to (1, 1024)

# Full pattern: conv base → GAP → flatten → classifier
model = nn.Sequential(
    nn.Conv2d(3, 1024, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d((1, 1)),   # GAP
    nn.Flatten(),                   # (batch, 1024)
    nn.Linear(1024, 10),            # classifier for 10 classes
)

In [ ]:
import torch
import torch.nn as nn

# The flattening step before a fully connected layer
x_volume = torch.randn(8, 256, 6, 6)   # batch of 8 feature maps (256 channels, 6x6 spatial)

flatten = nn.Flatten()                 # default: flattens dims 1 onwards
x_flat  = flatten(x_volume)
print(x_flat.shape)   # → torch.Size([8, 9216])  (8 × 256×6×6)

fc = nn.Linear(9216, 4096)
out = fc(x_flat)
print(out.shape)      # → torch.Size([8, 4096])

In [ ]:
import torch
import torch.nn as nn

# 1x1 convolution for channel-wise dimensionality reduction
# Reduces from 1024 to 256 channels at every spatial position
pointwise = nn.Conv2d(in_channels=1024, out_channels=256, kernel_size=1)

x = torch.randn(1, 1024, 14, 14)  # 1 image, 1024 channels, 14x14 spatial
out = pointwise(x)
print(out.shape)   # → torch.Size([1, 256, 14, 14])

# Compare parameter counts:
# FC layer mapping 1024-dim to 256-dim:
fc = nn.Linear(1024, 256)
print(sum(p.numel() for p in fc.parameters()))         # 1024*256 + 256 = 262400

# 1x1 conv mapping 1024 channels to 256 channels:
print(sum(p.numel() for p in pointwise.parameters()))  # same! 1024*256 + 256 = 262400

# They have the same number of parameters — but 1x1 conv can handle any H×W

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LeNet5(nn.Module):
    """LeNet-5 adapted for 32x32 grayscale input, 10 classes."""
    def __init__(self):
        super().__init__()
        # Convolutional base
        self.conv1 = nn.Conv2d(1, 6,  kernel_size=5)  # 32→28, 6 filters
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)  # 14→10, 16 filters
        # Classifier
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):             # x: (batch, 1, 32, 32)
        x = F.avg_pool2d(F.relu(self.conv1(x)), 2)  # → (batch, 6, 14, 14)
        x = F.avg_pool2d(F.relu(self.conv2(x)), 2)  # → (batch, 16, 5, 5)
        x = x.flatten(start_dim=1)                  # → (batch, 400)
        x = F.relu(self.fc1(x))                     # → (batch, 120)
        x = F.relu(self.fc2(x))                     # → (batch, 84)
        x = self.fc3(x)                             # → (batch, 10)
        return x

model = LeNet5()
x = torch.randn(4, 1, 32, 32) # 4 images, 1 channel, 32x32
print(model(x).shape)  # → torch.Size([4, 10])

# Count parameters
total = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total:,}')   # ≈ 61,706

In [ ]:
import torch
import torch.nn as nn

class AlexNet(nn.Module):
    """Simplified AlexNet for 224x224 RGB input, num_classes outputs."""
    def __init__(self, num_classes=1000):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=2),  # 224→55
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),                  # 55→27
            # Block 2
            nn.Conv2d(96, 256, kernel_size=5, padding=2),           # 27→27
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),                  # 27→13
            # Block 3
            nn.Conv2d(256, 384, kernel_size=3, padding=1),          # 13→13
            nn.ReLU(inplace=True),
            # Block 4
            nn.Conv2d(384, 384, kernel_size=3, padding=1),          # 13→13
            nn.ReLU(inplace=True),
            # Block 5
            nn.Conv2d(384, 256, kernel_size=3, padding=1),          # 13→13
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),                  # 13→6
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)         # → (batch, 256, 6, 6)
        x = x.flatten(start_dim=1)   # → (batch, 9216)
        return self.classifier(x)    # → (batch, num_classes)

model = AlexNet(num_classes=10)
total = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total:,}')   # ≈ 57 million

# Verify output shapes
x = torch.randn(2, 3, 224, 224)  # batch of 2 images (3 channels, 224x224)
out = model(x)
for layer in model.features:
    x = layer(x)
    print(f'{layer.__class__.__name__:20s}  →  {tuple(x.shape)}')

In [ ]:
import torch
import torch.nn as nn

class SmallCNN(nn.Module):
    """
    A compact CNN for image classification.
    Demonstrates all key components from Lecture 4.
    """
    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()

        # ── Convolutional base ────────────────────────────────────────
        # Pattern: Conv → ReLU → Conv → ReLU → MaxPool
        # Doubling channels after each pooling is a common design choice.
        self.features = nn.Sequential(
            # Block 1: 3 → 32 channels, spatial: 64 → 64 → 32
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1), # same conv
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # halve spatial dims

            # Block 2: 32 → 64 channels, spatial: 32 → 32 → 16
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3: 64 → 128 channels, spatial: 16 → 16 → 8
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # ── Classifier head ───────────────────────────────────────────
        # Global Average Pooling removes spatial dims entirely,
        # making the classifier work for any input image size.
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),  # → (batch, 128, 1, 1)
            nn.Flatten(),                  # → (batch, 128)
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),              # regularisation
            nn.Linear(64, num_classes),   # logits for each class
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# ── Training loop skeleton ────────────────────────────────────────────
model     = SmallCNN(in_channels=3, num_classes=10)
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn   = nn.CrossEntropyLoss()

# Dummy batch: 16 RGB images of size 64x64
images = torch.randn(16, 3, 64, 64)  # (batch, channels, H, W)
labels = torch.randint(0, 10, (16,)) # random integer labels in [0, 9]

# One training step
optimiser.zero_grad()           # 1. Clear gradients
logits = model(images)          # 2. Forward pass
loss   = loss_fn(logits, labels)# 3. Compute loss
loss.backward()                 # 4. Backward pass (compute gradients)
optimiser.step()                # 5. Update weights

print(f'Loss: {loss.item():.4f}')
print(f'Output shape: {logits.shape}')   # → (16, 10)

# Count parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')   # ≈ 102,602

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# ── Scenario 1: Feature extraction ───────────────────────────────────
# Load a ResNet-18 pre-trained on ImageNet
backbone = models.resnet18(weights='IMAGENET1K_V1')

# Freeze all parameters so they are not updated during training
for param in backbone.parameters():
    param.requires_grad = False

# Replace the final classifier with one suited to our task (e.g., 5 classes)
backbone.fc = nn.Linear(backbone.fc.in_features, 5)  # only this is trained

# Verify: only the new head has trainable parameters
trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
print(f'Trainable (feature extraction): {trainable:,}')   # = 2,565  (just the new FC layer)


# ── Scenario 2: Fine-tuning ───────────────────────────────────────────
backbone2 = models.resnet18(weights='IMAGENET1K_V1')
backbone2.fc = nn.Linear(backbone2.fc.in_features, 5)

# Unfreeze everything — use a small learning rate to avoid destroying
# the pre-trained features
optimiser = torch.optim.Adam(backbone2.parameters(), lr=1e-4)

print(f'Trainable (fine-tuning): {sum(p.numel() for p in backbone2.parameters() if p.requires_grad):,}')   # = 11,179,077 (all parameters trainable)